In [1]:
import pandas as pd

# 1. Load the data (Make sure the spelling matches your files exactly)
real_news = pd.read_csv('True.csv')
fake_news = pd.read_csv('Fake.csv')

# 2. Add a new column so the computer knows the answer key
real_news['label'] = 1  # 1 means Real News
fake_news['label'] = 0  # 0 means Fake News

# 3. Combine both datasets into one giant dataset
all_news = pd.concat([real_news, fake_news])

# 4. Shuffle the data so the model doesn't just memorize the order
all_news = all_news.sample(frac=1).reset_index(drop=True)

# 5. Display the first 5 rows to make sure it worked
all_news.head()

,title,text,subject,date,label
0,Saudi Arabia tightens security after Bahrain p...,DUBAI (Reuters) - Saudi Arabia s energy minist...,worldnews,"November 11, 2017",1
1,BREAKING #CNNLeak…James O’Keefe’s New Undercov...,James O Keefe has been at the forefront of new...,politics,"Feb 23, 2017",0
2,JUDGE REMOVES 1 YR OLD FROM HOME OF MARRIED LE...,Gay Mafia descends in 5 4 3 2 1 Given the curr...,left-news,"Nov 13, 2015",0
3,REMEMBER WHEN Democrat Operatives Were Caught ...,Watch:. @Jordanfabian If there is no #VoterFra...,politics,"Nov 28, 2016",0
4,Russia may widen designation for media outlets...,MOSCOW (Reuters) - Russia may decide to design...,worldnews,"December 21, 2017",1


In [2]:
import nltk
import re
from nltk.corpus import stopwords

# Download the official list of stop words (only needs to run once)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Make everything lowercase
    text = str(text).lower()
    
    # 2. Remove all punctuation (keeps only letters and spaces)
    text = re.sub(r'[^\w\s]', '', text)
    
    # 3. Remove stop words (the, is, and, etc.)
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]
    
    # Put the cleaned words back together into a sentence
    return ' '.join(cleaned_words)

# Combine the Title and the Text of the article so the AI can read both
all_news['full_text'] = all_news['title'] + " " + all_news['text']

print("Cleaning the text... (Please wait, this might take 1 to 3 minutes!)")

# Apply our cleaning function to every single article
all_news['cleaned_text'] = all_news['full_text'].apply(clean_text)

print("Done cleaning! Here is a before and after:")
all_news[['full_text', 'cleaned_text']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rooma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaning the text... (Please wait, this might take 1 to 3 minutes!)
Done cleaning! Here is a before and after:


,full_text,cleaned_text
0,Saudi Arabia tightens security after Bahrain p...,saudi arabia tightens security bahrain pipelin...
1,BREAKING #CNNLeak…James O’Keefe’s New Undercov...,breaking cnnleakjames okeefes new undercover r...
2,JUDGE REMOVES 1 YR OLD FROM HOME OF MARRIED LE...,judge removes 1 yr old home married lesbians s...
3,REMEMBER WHEN Democrat Operatives Were Caught ...,remember democrat operatives caught bragging v...
4,Russia may widen designation for media outlets...,russia may widen designation media outlets dee...


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define our study material (X) and our answer key (y)
X = all_news['cleaned_text'] # The scrubbed text
y = all_news['label']        # The 1s (Real) and 0s (Fake)

# 2. Split the data: 80% for learning, 20% for testing later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create the text-to-math translator
vectorizer = TfidfVectorizer()

# 4. Teach the translator the vocabulary and convert the training text into numbers
X_train_math = vectorizer.fit_transform(X_train)

# 5. Convert the testing text into numbers using the exact same vocabulary
X_test_math = vectorizer.transform(X_test)

print("Success! The text has been converted to math.")
print(f"We have {X_train_math.shape[0]} articles for training and {X_test_math.shape[0]} for testing.")

Success! The text has been converted to math.
We have 35918 articles for training and 8980 for testing.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Create the algorithm (the "brain")
model = LogisticRegression()

print("Training the model... (This will take a few seconds)")

# 2. Feed it the training math (X) and the answer key (y) so it can learn patterns
model.fit(X_train_math, y_train)

# 3. Give it the "final exam" using the 8,980 test articles it has never seen
predictions = model.predict(X_test_math)

# 4. Grade the exam by comparing its predictions to the real answer key (y_test)
score = accuracy_score(y_test, predictions)

print(f"Training Complete! Your model's accuracy is: {score * 100:.2f}%")

Training the model... (This will take a few seconds)
Training Complete! Your model's accuracy is: 98.76%


In [6]:
import joblib
import os

# 1. Make sure the saved_models folder is targeted correctly
save_path = '../saved_models/'
os.makedirs(save_path, exist_ok=True)

# 2. Save the trained algorithm (The Brain)
joblib.dump(model, save_path + 'model.pkl')

# 3. Save the vocabulary translator (The Dictionary)
joblib.dump(vectorizer, save_path + 'vectorizer.pkl')

print("Success! Your brain and translator are safely saved.")

Success! Your brain and translator are safely saved.
